In [ ]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session(default_bucket='sagemaker-congestion-bogota')
region = sagemaker_session.boto_region_name
role = 'arn:aws:iam::195419001736:role/SagemakerExecutionRole'
bucket_name = 'sagemaker-congestion-bogota'
print(region)
print(role)


In [ ]:
import os
import tarfile
import tempfile
import boto3

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'

def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    source_path = os.path.abspath(source_path)
    with tempfile.TemporaryDirectory() as tmpdir:
        tar_path = os.path.join(tmpdir, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_path, 'w:gz') as tar:
            tar.add(os.path.join(source_path, 'src'), arcname='src')
            tar.add(os.path.join(source_path, 'requirements-sagemaker.txt'), arcname='requirements-sagemaker.txt')
        s3 = boto3.client('s3', region_name=region)
        s3_uri = f's3://{bucket}/{s3_key_prefix}/{TAR_SOURCE_FILENAME}'
        s3.upload_file(tar_path, bucket, f'{s3_key_prefix}/{TAR_SOURCE_FILENAME}')
        return s3_uri

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/seed-time-artifact/code')
print(source_code_uri)


In [ ]:
image = '195419001736.dkr.ecr.us-east-1.amazonaws.com/congestion-bogota-sagemaker:temporal-features-20260331-194332'
job_name = f'congestion-bogota-seed-time-artifact-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}'

hyper_parameters = {
    'sagemaker_program': 'src/rebalance_sampling.py',
    'sagemaker_submit_directory': source_code_uri,
    'input-seq-len': '150',
    'output-seq-len': '50',
    'node-batch-size': '500',
    'train-ratio': '0.8',
    'include-weather': 'True',
    'include-velocity-target': 'True',
    'use-minimal-datasets': 'True',
    'high-level-threshold': '3.0',
    'bucket-quantiles': '0.5,0.75',
    'max-seed-candidates': '200',
    'max-time-indices': '1000',
    'progress-log-every': '5000',
    'output-parquet': '/opt/ml/model/seed_time_bucket_rows.parquet',
    'output-metadata': '/opt/ml/model/seed_time_bucket_metadata.json',
}

print(job_name)


In [ ]:
import boto3

sm = boto3.client('sagemaker', region_name=region)
NUMERIC_REGEX = r'([-+0-9eE\\.]+)'

metric_names = [
    'num_time_windows',
    'train_time_indices',
    'seed_candidates',
    'node_batch_size',
    'input_seq_len',
    'output_seq_len',
    'pairs_evaluated',
    'seeds_completed',
    'partial_score_mean',
    'partial_score_min',
    'partial_score_max',
    'partial_low_ratio',
    'partial_medium_ratio',
    'partial_high_ratio',
    'num_seeds',
    'num_time_indices',
    'score_min',
    'score_mean',
    'score_max',
    'bucket_low_count',
    'bucket_medium_count',
    'bucket_high_count',
]

metric_definitions_payload = [
    {'Name': name, 'Regex': rf'{name}={NUMERIC_REGEX}'}
    for name in metric_names
]

training_job_request = {
    'TrainingJobName': job_name,
    'RoleArn': role,
    'AlgorithmSpecification': {
        'TrainingImage': image,
        'TrainingInputMode': 'File',
        'MetricDefinitions': metric_definitions_payload,
        'EnableSageMakerMetricsTimeSeries': True,
    },
    'HyperParameters': hyper_parameters,
    'InputDataConfig': [
        {
            'ChannelName': 'train',
            'DataSource': {
                'FileSystemDataSource': {
                    'FileSystemId': 'fs-099e29197c0a3e353',
                    'FileSystemAccessMode': 'ro',
                    'FileSystemType': 'EFS',
                    'DirectoryPath': '/data/datos_procesados',
                }
            },
        },
    ],
    'OutputDataConfig': {
        'S3OutputPath': 's3://sagemaker-congestion-bogota/congestion-bogota/seed-time-artifact',
    },
    'ResourceConfig': {
        'InstanceType': 'ml.r7i.8xlarge',
        'InstanceCount': 1,
        'VolumeSizeInGB': 100,
    },
    'StoppingCondition': {
        'MaxRuntimeInSeconds': 7200,
    },
    'VpcConfig': {
        'SecurityGroupIds': ['sg-0962698c0c8352ddd', 'sg-03590ecb70a82d570'],
        'Subnets': ['subnet-017c79447d5151efe'],
    },
}

response = sm.create_training_job(**training_job_request)
print(response['TrainingJobArn'])


In [ ]:
import boto3

sm = boto3.client('sagemaker', region_name=region)
desc = sm.describe_training_job(TrainingJobName=job_name)
print(desc['TrainingJobStatus'])
print(desc['SecondaryStatus'])
print(desc.get('FailureReason'))


In [ ]:
desc = boto3.client('sagemaker', region_name=region).describe_training_job(TrainingJobName=job_name)
for metric in desc.get('FinalMetricDataList', []):
    print(metric['MetricName'], metric['Value'])
